In [1]:
import subprocess
import os
import csv
import pandas as pd
from bayes_opt import BayesianOptimization
from pathlib import Path
import numpy as np
from simulation_functions import call_vot_simulation, intialise_java, call_stt_simulation,stats_test,STT_INIT_POINTS, STT_N_ITER, VOT_INIT_POINTS, VOT_N_ITER
from scipy import stats

VOT config

In [2]:
#VOT config
defaultAlpha_bounds  = (-5,5)
default_beta_bounds = (-1, -0.001)
vot_pbounds = {'alphaWalk' :(0,0) ,
           'alphaBike' : defaultAlpha_bounds,
           'alphaCarDriver' : defaultAlpha_bounds,
           'alphaCarPassenger' : defaultAlpha_bounds,
           'alphaBus' : defaultAlpha_bounds,
           'alphaTrain' : defaultAlpha_bounds,
           'betaChangesTransport': default_beta_bounds,
           'betaCostLow':(-0.2,-0.2) ,
           'betaCostMed': (-0.2,-0.2),
           'betaCostHigh': (-0.2,-0.2),
           'weightWalk': (1,3),
           'weightWait': (1,3),
           'weightFeeder': (1,3), 
           'weightVotCosts' : (1,1),
           'weightTangibleCosts' : (1,1)
           } 

STT config

In [3]:
stt_pbounds = {
    'alphaWalk' :(0,0) ,
    'alphaBike' : defaultAlpha_bounds,
    'alphaCarDriver' : defaultAlpha_bounds,
    'alphaCarPassenger' : defaultAlpha_bounds,
    'alphaBus' : defaultAlpha_bounds,
    'alphaTrain' : defaultAlpha_bounds,
    'betaTimeWalk' : (-0.04,-0.04),
    'betaTimeBike' : (-0.03,-0.03),
    'betaTimeCarDriver': (-0.02,-0.02),
    'betaTimeCarPassenger' : (-0.02,-0.02),
    'betaTimeBus' : (-0.02,-0.02),
    'betaTimeTrain' : (-0.02,-0.02),
    'betaCostCarDriver': (-0.15,-0.15),
    'betaCostCarPassenger' : (-0.15,-0.15),
    'betaCostBus': (-0.1,-0.1),
    'betaCostTrain': (-0.1,-0.1),
    'betaTimeWalkTransport': (-0.03,-0.03),
    'betaChangesTransport': (-0.3,-0.3)}

In [22]:
stt_gen_synth_pop_baseline = [1.538359177843295,
 1.5295917791690679,
 1.5348367663998912,
 1.5345692372273643,
 1.5331592512372867,
 1.5217034952544148,
 1.5266326185511052,
 1.5334169124971047,
 1.5360137686306476,
 1.5339017221565026,
 1.5362097781749502,
 1.5268521109958817,
 1.5339477732745261,
 1.5272698871615034,
 1.5366142065675472,
 1.5363138577063769,
 1.5296998668830282,
 1.5197286106223606,
 1.5389714190118928,
 1.5313439434581202,
 1.537382448574118,
 1.5343533532165867,
 1.533794502058735,
 1.5379044402275153,
 1.540476837032657,
 1.5442550528088403,
 1.5314370912595778,
 1.5282256003186678,
 1.529788503142131,
 1.5278669293628444]

vot_gen_synth_pop_baseline = [1.385527667908908,
 1.3928263136401784,
 1.3789626118986595,
 1.3877472402566997,
 1.3871102011278413,
 1.385742628118824,
 1.3912868550950295,
 1.3872805327363473,
 1.3837930639491043,
 1.3895217968284068,
 1.3875468568014604,
 1.378848603328448,
 1.3894661348423816,
 1.388160109275813,
 1.3871204285727328,
 1.390915593919979,
 1.3787646467724206,
 1.384115813260929,
 1.3823050198239224,
 1.3847381859154229,
 1.3816234011419537,
 1.3867555230241397,
 1.3876604079221397,
 1.3822073266194879,
 1.3825751517648885,
 1.3835028373387503,
 1.3806274629010047,
 1.3837965494208504,
 1.389955262359422,
 1.377912990757942]

stt_base_pop_baseline = [8.078856646164647,
 8.069551887294185,
 8.075341126236319,
 8.077668179308883,
 8.072089261165532,
 8.08434725328873,
 8.070135626168561,
 8.07003999884651,
 8.074229639153051,
 8.073871153762488,
 8.076644709455191,
 8.067903638080667,
 8.074645258436167,
 8.07695697572408,
 8.069666279765226,
 8.073238056549226,
 8.075901298340282,
 8.074258667977556,
 8.066514722376704,
 8.072437841187186,
 8.07148870370448,
 8.069260779586159,
 8.067953580518305,
 8.074462078074248,
 8.072890298665042,
 8.076124680057342,
 8.07104170586535,
 8.066528599568002,
 8.070791356185456,
 8.081042173765757]

vot_base_pop_baseline = [7.931848576119684,
 7.931369714896462,
 7.9433848896215995,
 7.937736686895536,
 7.941242398531084,
 7.936923206985956,
 7.936669645073594,
 7.940973608526627,
 7.936207185980619,
 7.936291631277916,
 7.937923289418983,
 7.941331709310874,
 7.93954170300512,
 7.936830089802029,
 7.939676996588786,
 7.9394618950690505,
 7.939708733929694,
 7.941668216327173,
 7.941204779953899,
 7.943912118112101,
 7.944579406069159,
 7.941662315056593,
 7.933800474344775,
 7.935876751446223,
 7.9376907386207165,
 7.935850335717845,
 7.939215465650584,
 7.94108731535453,
 7.934474803549772,
 7.94179396788631]

In [5]:
intialise_java()

Java Version Output:
 openjdk version "11.0.20.1" 2023-08-24 LTS
OpenJDK Runtime Environment Microsoft-8297088 (build 11.0.20.1+1-LTS)
OpenJDK 64-Bit Server VM Microsoft-8297088 (build 11.0.20.1+1-LTS, mixed mode)



RQ3.1

In [ ]:

rq_3_1_config = {
    "config_file": "src/main/resources/config_DHZW_rq3-1.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq3-1.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq3-1.csv',
    "other_args" : "--use_random_seed true"
}
rq3_1_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_simulation(config=rq_3_1_config, **params),
    pbounds=stt_pbounds,
    random_state=1,
)

rq3_1_optimiser.maximize(
    init_points=STT_INIT_POINTS,
    n_iter=STT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaCo... | betaCo... | betaCo... | betaCo... | betaTi... | betaCh... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -33.00594 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 2         | -16.77692 | 0.0       | -4.434806 | 2.6113059 | -2.915663 | 2.7732349 | 2.1649444 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |


In [ ]:
rq_3_1_config_no_seed = rq_3_1_config
rq_3_1_config_no_seed["other_args"] = "--use_random_seed false"
rq_3_1_results = []
for i in range(0,30):
    rq_3_1_results.append(-call_stt_simulation(config=rq_3_1_config_no_seed,**rq3_1_optimiser.max['params']))


In [9]:
rq_3_1_results

[16.78286328273859, 16.77994877385295]

RQ3.2

In [ ]:

rq_3_2_config = {
    "config_file": "src/main/resources/config_DHZW_rq3-2.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq3-2.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq3-2.csv',
    "other_args" : "--use_random_seed true"
}
rq_3_2_optimiser = BayesianOptimization(
    f=lambda **params: call_vot_simulation(config=rq_3_2_config, **params),
    pbounds=vot_pbounds,
    random_state=1,
)

rq_3_2_optimiser.maximize(
    init_points=VOT_INIT_POINTS,
    n_iter=VOT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaCo... | betaCo... | betaCo... | weight... | weight... | weight... | weight... | weight... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -32.91104 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.813926 | -0.2      | -0.2      | -0.2      | 1.8383890 | 2.3704390 | 1.4089044 | 1.0       | 1.0       |
| 2         | -33.14278 | 0.0       | 4.0051177 | 2.0127188 | -1.937694 | 0.7297246 | 2.0292742 | -0.865947 | -0.2      | -0.2      | -0.2      | 2.8397151 | 1.7312530 | 1.6453775 | 1.0       | 1.0       |


In [ ]:
rq_3_2_config_no_seed = rq_3_2_config
rq_3_2_config_no_seed["other_args"] = "--use_random_seed false"
rq_3_2_results = []
for i in range(0,30):
    rq_3_2_results.append(- call_vot_simulation(config=rq_3_2_config_no_seed,**rq_3_2_optimiser.max['params']))

In [12]:
rq_3_2_results

[32.902305020120636, 32.90254480486694]

RQ3.3

In [ ]:

rq_3_3_config = {
    "config_file": "src/main/resources/config_DHZW_rq3-3.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq3-3.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq3-3.csv',
    "other_args" : "--use_random_seed true"
}
rq3_3_optimiser = BayesianOptimization(
    f=lambda **params: call_stt_simulation(config=rq_3_3_config, **params),
    pbounds=stt_pbounds,
    random_state=1,
)

rq3_3_optimiser.maximize(
    init_points=STT_INIT_POINTS,
    n_iter=STT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaTi... | betaCo... | betaCo... | betaCo... | betaCo... | betaTi... | betaCh... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -33.11285 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |
| 2         | -13.70648 | 0.0       | -4.434806 | 2.6113059 | -2.915663 | 2.7732349 | 2.1649444 | -0.04     | -0.03     | -0.02     | -0.02     | -0.02     | -0.02     | -0.15     | -0.15     | -0.1      | -0.1      | -0.03     | -0.3      |


In [ ]:
rq_3_config_no_seed = rq_3_3_config
rq_3_config_no_seed["other_args"] = "--use_random_seed false"
rq_3_3_results = []
for i in range(0,30):
    rq_3_3_results.append(-call_stt_simulation(config=rq_3_config_no_seed,**rq3_3_optimiser.max['params']))

In [15]:
rq_3_3_results

[13.68800476578849, 13.696204937395857]

RQ3.4

In [ ]:
rq_3_4_config = {
    "config_file": "src/main/resources/config_DHZW_rq3-4.toml",
    "parameterset": "src/main/resources/baseline_parameterset/parameterset_rq3-4.csv",
    "parameterset_write_folder" : "../7-simulation-Sim-2APL/",
    "output_path": 'src/main/resources/baseline_parameterset/output-rq3-4.csv',
    "other_args" : "--use_random_seed true"
}
rq_3_4_optimiser = BayesianOptimization(
    f=lambda **params: call_vot_simulation(config=rq_3_4_config, **params),
    pbounds=vot_pbounds,
    random_state=1,
)

rq_3_4_optimiser.maximize(
    init_points=VOT_INIT_POINTS,
    n_iter=VOT_N_ITER
)

|   iter    |  target   | alphaWalk | alphaBike | alphaC... | alphaC... | alphaBus  | alphaT... | betaCh... | betaCo... | betaCo... | betaCo... | weight... | weight... | weight... | weight... | weight... |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| 1         | -32.70824 | 0.0       | 2.2032449 | -4.998856 | -1.976674 | -3.532441 | -4.076614 | -0.813926 | -0.2      | -0.2      | -0.2      | 1.8383890 | 2.3704390 | 1.4089044 | 1.0       | 1.0       |
| 2         | -32.77280 | 0.0       | 4.0051177 | 2.0127188 | -1.937694 | 0.7297246 | 2.0292742 | -0.865947 | -0.2      | -0.2      | -0.2      | 2.8397151 | 1.7312530 | 1.6453775 | 1.0       | 1.0       |


In [ ]:
rq_3_4_config_no_seed = rq_3_4_config
rq_3_4_config_no_seed["other_args"] = "--use_random_seed false"
rq_3_4_results = []
for i in range(0,30):
    rq_3_4_results.append(- call_vot_simulation(config=rq_3_4_config_no_seed,**rq_3_4_optimiser.max['params']))

In [18]:
rq_3_4_results

[32.69838540456914, 32.708819932104205]

Significance

In [20]:
stats_test(stt_base_pop_baseline, rq_3_1_results)

P-value: 3.5468865391030315e-83
Statistically significant difference.
t-statistic: -2876.6718
p-value: 0.0000
95% CI of the difference: [-8.7116, -8.7048]


In [23]:
stats_test(vot_base_pop_baseline, rq_3_2_results)

P-value: 7.952301880027329e-100
Statistically significant difference.
t-statistic: -10324.5113
p-value: 0.0000
95% CI of the difference: [-24.9650, -24.9625]


In [24]:
stats_test(stt_gen_synth_pop_baseline, rq_3_3_results)

P-value: 4.722625674198308e-84
Statistically significant difference.
t-statistic: -3076.6579
p-value: 0.0000
95% CI of the difference: [-12.1679, -12.1507]


In [25]:
stats_test(vot_gen_synth_pop_baseline, rq_3_4_results)

P-value: 9.24862553225042e-100
Statistically significant difference.
t-statistic: -10272.6706
p-value: 0.0000
95% CI of the difference: [-31.3291, -31.3076]
